In [1]:
!pip install ultralytics


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install roboflow

  Using cached roboflow-1.3.8-py3-none-any.whl.metadata (11 kB)
  Using cached idna-3.7-py3-none-any.whl.metadata (9.9 kB)
  Using cached opencv_python_headless-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached roboflow-1.3.8-py3-none-any.whl (207 kB)
Using cached idna-3.7-py3-none-any.whl (66 kB)
Using cached opencv_python_headless-4.10.0.84-cp37-abi3-win_amd64.whl (38.8 MB)
Using cached typer-0.25.1-py3-none-any.whl (58 kB)
Using cached rich-15.0.0-py3-none-any.whl (310 kB)
Using cached markdown_it_py-4.2.0-py3-none-any.whl (91 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install load_dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import ultralytics
from IPython.display import display, Image, clear_output
from ultralytics import YOLO
from cv2 import line
from PIL import Image
from roboflow import Roboflow

ultralytics.checks()
clear_output()

In [5]:
DRIVE_URL = '/content/drive'
LOCAL_PATH = os.getcwd()

In [6]:
def detect_environment():
    try:
        ipython_env = str(get_ipython())

        if 'google.colab' in ipython_env:
            return "Google Colab"
        elif 'zmqshell' in ipython_env:
            return "Jupyter Notebook Local"
        else:
            return "Terminal interactiva de Python"
    except NameError:
        return "Script de Python estándar (.py)"


print(f"Entorno detectado: {detect_environment()}")

Entorno detectado: Jupyter Notebook Local


In [7]:
if detect_environment() != "Google Colab":
    BASE = LOCAL_PATH
else:
    BASE = DRIVE_URL
    from google.colab import drive
    from google.colab.patches import cv2_imshow
    drive.mount('/content/drive')
    from google.colab import userdata

In [8]:
BASE_IMGS = os.path.join(BASE, "imgs")
BASE_MODELS = os.path.join(BASE, "models")

In [9]:
model_n= YOLO(os.path.join(BASE_MODELS, "yolov8n.pt"))

In [10]:
result = model_n.predict(os.path.join(BASE_IMGS, "imagen_yolo.jpg"))

FileNotFoundError: C:\Users\Sebastian-EDU\Documents\GitHub\EvolutionaryComputation\RoboFlow\imgs\imagen_yolo.jpg does not exist

In [ ]:
result[0].names

In [ ]:
result[0].boxes

In [ ]:
for r in result:
  im_array= r.plot(line_width=2)
  im= Image.fromarray(im_array[...,::-1])
  cv2_imshow(im_array)
  im.save(os.path.join(BASE_IMGS, "detection.jpg"))

# Entrenamiento de Modelo

In [ ]:
model_l= YOLO(os.path.join(BASE_MODELS, "yolov8l.pt"))

In [ ]:
def setup_roboflow_credentials():
    env = detect_environment()
    api_key = None

    if env == "colab":
        try:
            api_key = userdata.get('ROBOFLOW_API_KEY')
            print("Credenciales de Roboflow cargadas desde Secretos de Colab.")
        except Exception:
            print("Configura 'ROBOFLOW_API_KEY' en los secretos de Colab.")
    else:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            print("python-dotenv no está instalado. Usando variables de entorno del sistema...")

        api_key = os.environ.get('ROBOFLOW_API_KEY')

        if api_key:
            print("Credenciales de Roboflow cargadas desde variables de entorno (.env).")
        else:
            print("No se encontró 'ROBOFLOW_API_KEY' en el entorno local.")

    return api_key

In [ ]:
rf_api_key = setup_roboflow_credentials()

In [ ]:
if rf_api_key:
    try:
        rf = Roboflow(api_key=rf_api_key)

        workspace_name = 'yolov8-rdmbf'
        project_name = 'bccd-f0tcy'

        print(f"Descargando proyecto '{project_name}' del workspace '{workspace_name}'...")

        project = rf.workspace(workspace_name).project(project_name)
        version = project.version(1)
        dataset = version.download("yolov8")

        print("Dataset descargado correctamente en:", dataset.location)

    except Exception as e:
        print(f"Ocurrió un error al conectar con Roboflow: {e}")
else:
    print("Ejecución detenida: No hay credenciales válidas para Roboflow.")